<h1> Notebook Summary </h1>

Partially adapted from : ...
TODO: Add a summary of the notebook here. Also include instructions on creating a venv

<h1> Retreiving the GEDI data from NASA Earthdata </h1>

<h3> We first need to import required dependencies </h3>

In [ ]:
from harmony import BBox, Client, Collection, Request, CapabilitiesRequest # NASA Harmony
import earthaccess # NASA Earthdata Login
from datetime import datetime
import os
import json
import requests
import tempfile
import h5py
import pandas as pd

<h3> Login to NASA Earthdata </h3>

You may need to register for a profile on NASA Earthdata to do this. Here is the link: https://urs.earthdata.nasa.gov/users/new

Once you are logged in, click on "Generate Token" to create an API token to allow you to access Earthdata. Save this token somewhere safe. 

In [5]:
auth = earthaccess.login(persist=True)

<h3> Upload GeoJSON of area of interest </h3>

To retrieve the area of interest, we will need a GeoJSON file containing a bounding box of the area of interest. Once you have this file, store it **in the same folder** as this notebook. 

You may only have **ONE GeoJSON file in this folder**, and it **must be named with the following convention: [place_name].geojson.**

In [ ]:
curr_dir = os.getcwd()
geojson_files = [f for f in os.listdir(curr_dir) if f.endswith('.geojson')]

if len(geojson_files) == 0:
    print("No GeoJSON file found in the current directory. Please add a GeoJSON file and try again.")
elif len(geojson_files) > 1:
    print("Multiple GeoJSON files found in the current directory. Please ensure there is only one GeoJSON file and try again.")
else:
    geojson_file_path = os.path.join(curr_dir, geojson_files[0])
    my_site = geojson_file_path
    try:
        with open(my_site) as f:
            geojson_polygon = json.load(f)
        print("GeoJSON loaded successfully: ", my_site)
    except json.JSONDecodeError:
        print(f"Invalid GeoJSON format in file: {my_site}")

<h3> Select the dates for retreival </h3>

GEDI data is available during the following time periods:
*   April 2019 - March 2023
*   April 2024 - Present

If doing multiple retrievals, it is recommended that you don't do a retrrieval for more than 730 days/2 years at a time to ensure all data is retrieved successfully and to avoid timeouts.

In [ ]:
start_str = input("Enter start date (YYYY-MM-DD): ")
stop_str = input("Enter stop date (YYYY-MM-DD): ")

# Convert to datetime 
try:
    temporal_range = {
        'start': datetime.strptime(start_str, "%Y-%m-%d"),
        'stop': datetime.strptime(stop_str, "%Y-%m-%d")
    }
    print("Start date: ", temporal_range['start'])
    print("Stop date: ", temporal_range['stop'])
except ValueError:
    print("Invalid date format. Please try again and use YYYY-MM-DD.")

<h3> [Optional] Select the output folder for your results </h3>

If you do not want the outputted CSV of GEDI data to be stored in the same folder as this notebook, you can specify an output folder here. If you do not specify an output folder, the CSV will be stored in the same folder as this notebook.

The output folder should be FROM THE ROOT of this repository. In VSCode, you can right click on the desired output folder and select "Copy Relative Path" to get the correct path.

If you keep recieving errors, just do not specify an output folder and the results will be saved in the same folder as this notebook. This is the simplest.

In [ ]:
output_folder = input("Enter the path to the output folder where you wish to save your results, or press Enter to use the current folder:")

if output_folder.strip() == "":
    output_folder = curr_dir
else:
    if not os.path.exists(output_folder):
        print(f"Output folder '{output_folder}' does not exist. Please create the folder and try again.")
        exit(1)

In [ ]:
print(f"GEDI data will be saved in CSV format in: {output_folder}")

<h3> Call Harmony to retrieve the GEDI data h5 files</h3>

**NOTE: ** It is likely that your request will pause after a bit (unless it is a very small date range or area of interest). To resume, you will need to run the chunk below the following chunk containing:

`task_json = harmony_client.resume(task)`
`task_json = harmony_client.result_json(task, show_progress=True)`


In [ ]:
harmony_client = Client(auth=(auth.username, auth.password))
capabilities_request = CapabilitiesRequest(short_name='GEDI02_A')
capabilities = harmony_client.submit(capabilities_request)
concept_id = capabilities['conceptId']

request = Request(
    collection = Collection(id=concept_id),
    shape = my_site,
    temporal = temporal_range
)
task = harmony_client.submit(request)
print(f'Harmony request ID: {task}')
print(f'Processing your Harmony request:')

task_json = harmony_client.result_json(task, show_progress=True)

In [ ]:
# Run this in the job is paused by harmony
task_json = harmony_client.resume(task)
task_json = harmony_client.result_json(task, show_progress=True)

<h3> Save the retrieved h5 files in a txt file </h3>

The h5 files contain api links to the actual data. We will save these api links in a txt file to make it easier to download the data using a script. The h5 files will be saved in the same folder as this notebook. 

In [ ]:
h5_files = [link['href'] for link in task_json['links'] if link['href'].endswith('.h5')]
with open(os.path.join(os.getcwd(), 'h5_files.txt'), 'w') as f:
    for h5_file in h5_files:
        f.write(h5_file + '\n')

print("H5 file links saved to h5_files.txt")

<h3> Ask about quality and other filtering options </h3>

Sensitivity: This is a measure of the quality/accuracy of the data. The higher your threshold, the more accurate your data will be, but you will also get less data. It's recommended to set your threshold between 0.5 and 0.9. 


RH95: This is the 95th percentile of tree heights in a specific area that the GEDI data is collected from (since every point is ~20m in diameter). Setting a rh95 threshold filters out data that may be noisy. For instance, having negative rh95 values is not possible as trees cant be negative heights, so setting a rh95 threshold of 0 or higher can help filter out noisy data. Setting a rh95 threshold of 2 or higher can help filter out data that may be from areas with very short vegetation (such as grasslands).

Nighttime Data: This is data that is collected at night. The benefit of using data from only the night is that it is not affected by solar noise at similar wavelengths. However, though much more accurate, this will get rid of a lot of data as it only keeps points with solar elevation < 0.

In [ ]:
min_sensitivity_threshold = float(input("Enter the minimum sensitivity threshold (recommended between 0.5 and 0.9): "))
min_rh95_threshold = float(input("Enter the minimum rh95 threshold (decimal between 0 and 5): "))
nighttime_data_only = input("Do you want to include only nighttime data? (Y/N): ")

nighttime_data_only = bool(nighttime_data_only.strip().upper() == 'Y')

if min_sensitivity_threshold < 0 or min_sensitivity_threshold > 1:
    print("Invalid sensitivity threshold. Please re-run and enter a decimal value between 0 and 1.")
    exit(1)

print(f"Minimum sensitivity threshold set to: {min_sensitivity_threshold}")
print(f"Minimum rh95 threshold set to: {min_rh95_threshold}")
print(f"Nighttime data only: {nighttime_data_only}")

<h3> Create function to filter the data based on the above options </h3>

Adopted from ____________DO HEREE________

In [ ]:
def extract_gedi_rh_metrics_from_urls(h5_files_list, output_csv_file, beams=None,
                                      min_sensitivity=0.9, min_rh95=1,
                                      night=False):
    """
    Reads a list of HTTPS GEDI HDF5 file URLs, downloads each file completely,
    extracts RH metrics for specified beams, filters by sensitivity, RH95,
    and optionally by solar elevation (nighttime), and saves all shots to a CSV.

    Parameters
    ----------
    h5_files_list : str
        Path to a text file containing one HTTPS URL per line.
    output_csv_file : str
        Output CSV file path.
    beams : list, optional
        List of beam names to extract. Defaults to the four full-power beams.
    min_sensitivity : float, optional
        Minimum acceptable sensitivity (default: 0.95)
    min_rh95 : float, optional
        Minimum acceptable RH95 value (default: 0)
    night : bool, optional
        If True, select only shots where solar_elevation < 0 (nighttime)
    """
    if beams is None:
        beams = ['BEAM0101', 'BEAM0110', 'BEAM1000', 'BEAM1011']

    # Read list of URLs
    with open(h5_files_list, 'r') as f:
        h5_urls = [line.strip() for line in f if line.strip()]

    records = []

    for url in h5_urls:
        print(f"\n📥 Downloading {url} ...")
        tmp_path = None

        try:
            response = requests.get(url, timeout=300) # works because earthaccess.login(persist=True) stored creds in ~/.netrc
            response.raise_for_status()
            
            # save file to temp directory
            with tempfile.NamedTemporaryFile(suffix=".h5", delete=False) as tmp:
                tmp.write(response.content)
                tmp_path = tmp.name

            with h5py.File(tmp_path, 'r') as f:
                for beam_name in beams:
                    if beam_name not in f:
                        print(f"  ⚠️ Beam {beam_name} not found in {url}")
                        continue

                    beam = f[beam_name]

                    # Check datasets required for GEDI L2A v2.1+
                    required = ['lat_lowestmode', 'lon_lowestmode', 'rh', 'shot_number', 'solar_elevation']
                    missing = [k for k in required if k not in beam]
                    if len(missing) > 0:
                        print(f"  ⚠️ Missing the following data for {beam_name} in {url}: \n{missing}")
                        continue

                    # Try to access sensitivity safely
                    sensitivity = None
                    if 'QA' in beam and 'sensitivity' in beam['QA']:
                        sensitivity = beam['QA/sensitivity'][:]
                    elif 'sensitivity' in beam:
                        sensitivity = beam['sensitivity'][:]
                    else:
                        print(f"  ⚠️ Sensitivity missing in {beam_name} in {url}, skipping beam")
                        continue

                    lat = beam['lat_lowestmode'][:]
                    lon = beam['lon_lowestmode'][:]
                    shot_num = beam['shot_number'][:]
                    rh = beam['rh'][:]
                    solar = beam['solar_elevation'][:]

                    # Extract selected RH metrics
                    rh25 = rh[:, 25]
                    rh50 = rh[:, 50]
                    rh75 = rh[:, 75]
                    rh95 = rh[:, 95]

                    # Apply filters
                    mask = (sensitivity >= min_sensitivity) & (rh95 >= min_rh95)
                    if night:
                        mask &= (solar < 0)

                    if not mask.any():
                        print(f"  NOTE: ⚙️ No shots passed filters in {beam_name} in {url}")
                        continue

                    df = pd.DataFrame({
                        "source_url": url,
                        "beam": beam_name,
                        "shot_number": shot_num[mask],
                        "latitude": lat[mask],
                        "longitude": lon[mask],
                        "rh25": rh25[mask],
                        "rh50": rh50[mask],
                        "rh75": rh75[mask],
                        "rh95": rh95[mask],
                        "sensitivity": sensitivity[mask],
                        "solar_elevation": solar[mask]
                    })

                    records.append(df)

        except requests.exceptions.RequestException as e:
            print(f"❌ Download failed for {url}: {e}")
        except Exception as e:
            print(f"❌ Error processing {url}: {e}")
        finally:
            if 'tmp_path' in locals() and os.path.exists(tmp_path):
                os.remove(tmp_path)

    # Combine all dataframes
    if records:
        shots_df = pd.concat(records, ignore_index=True)
        shots_df.to_csv(csv_file, index=False)
        print(f"\n✅ Total shots saved: {len(shots_df):,}")
        print(f"✅ Output file: {csv_file}")
    else:
        print("\n⚠️ No valid shots extracted.")


<h3> Run the above function on the retrieved h5 files with the specified filtering options and save the results in a CSV </h3>

In [ ]:
h5_files = os.path.join(os.getcwd(), 'h5_files.txt')
out_csv_file = os.path.join(output_folder, 'gedi_shots.csv')

#create the file first
try:
    with open(out_csv_file, 'w') as f:
        f.write('') # just create an empty file
except Exception as e:
    with open(os.path.join(curr_dir, 'gedi_shots.csv'), 'w') as f:
        f.write('')
    out_csv_file = os.path.join(curr_dir, 'gedi_shots.csv')
    print(f"WARNING: Could not create output file in specified folder. Defaulting to current directory. Error details: {e}")

#List of Full Power Beams
full_power_beams = ['BEAM0101', 'BEAM0110', 'BEAM1000', 'BEAM1011']

#Run the function to filter H5 file and extract RH metrics
extract_gedi_rh_metrics_from_urls(h5_files, out_csv_file, 
                                  beams=full_power_beams,
                                  min_sensitivity=min_sensitivity_threshold, 
                                  min_rh95=min_rh95_threshold,
                                  night=nighttime_data_only)